# 01 GBDT
## 1 首先specify hyperparameters, 根据paper里面对于xgboost的调参参考，得到了GBDT的几个参数建议
《Comparing hyperparameter tuning methods in machine learning based urban building energy modeling: A study in Chicago - ScienceDirect》
    'n_estimators': [4168],
    'learning_rate': loguniform(0.002, 0.355),
    'subsample': uniform(0.545, 0.413),
    'max_depth' : randint(5, 9),

In [4]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RepeatedKFold
from scipy.stats import randint, uniform, loguniform

# 📁 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map'

# 🔧 变量定义
target_vars = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']

# 📊 保存结果
all_results = []
pdp_records = []
r2_comparison = []

# 🔍 Randomized Search 的分布（based on q0.05–q0.95）
param_dist = {
    # n_estimators = n_rounds(R) in xgboost-
    'n_estimators': [4168],
    # 0000000 learning_rate = shrinkage
    'learning_rate': loguniform(0.002, 0.355),
    # 0000000 subsample = subsample in xgboost#[0.545,0.958)  loc = 0.545 pos = 0.958-0.545 =0.413
    'subsample': uniform(0.545, 0.413),
    # 0000000  00 max_depth = Def.O = 13  5.6-14 -> 5-14
    'max_depth' : randint(5, 9)
    }

'''
'min_samples_split': uniform (1.295, 5.689),
# max_features ≈ colsample_bytree in xgboost,
'max_features': uniform(0.419, 0.445),
# ccp_alpha = alpha in xgboost
'ccp_alpha' : [1.113]
'''

# === 主循环 ===
for filename in os.listdir(grid_folder):
    if filename.endswith('_clean.shp'):
        input_path = os.path.join(grid_folder, filename)
        match = re.search(r'(\d{3,5})m', filename)
        grid_size = match.group(1)

        gdf = gpd.read_file(input_path)
        gdf_clean = gdf.replace([np.inf, -np.inf], np.nan).dropna(subset=target_vars + explanatory_vars)

        for target in target_vars:
            X = gdf_clean[explanatory_vars]
            y = gdf_clean[target]

            # 🔀 数据划分
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

            '''
            # 换成random search  定义log distribution
            # ----> 换成repeated cv ---- #
            # 至少20次 repeated number
            '''
            gbdt = GradientBoostingRegressor(random_state=0)
            cv = RepeatedKFold(n_splits=5, n_repeats=4, random_state=0)
            search = RandomizedSearchCV(
                estimator=gbdt,
                param_distributions=param_dist,
                n_iter= 200,
                scoring='r2',
                cv=cv,
                verbose=2,
                n_jobs=-1,
                random_state=0
            )

            search.fit(X_train, y_train)

            # ✅ 使用测试集评估
            best_model = search.best_estimator_
            y_train_pred = best_model.predict(X_train)
            y_test_pred = best_model.predict(X_test)

            r2_train = best_model.score(X_train, y_train)
            r2_test = r2_score(y_test, y_test_pred)

            rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
            rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

            print(f"✅ {filename} | {target} 最佳参数: {search.best_params_} | R²_train={r2_train:.3f} | R²_test={r2_test:.3f}")

            for var, importance in zip(explanatory_vars, best_model.feature_importances_):
                all_results.append({
                    'GridSize': grid_size,
                    'Target': target,
                    'Feature': var,
                    'FeatureImportance_TrainModel': round(importance, 4),
                    'Train_R2': round(r2_train, 4),
                    'Train_RMSE': round(rmse_train, 4),
                    'Test_R2': round(r2_test, 4),
                    'Test_RMSE': round(rmse_test, 4),
                    **search.best_params_
                })
                r2_comparison.append({
                    'GridSize': grid_size,
                    'Target': target,
                    'Train_R2': round(r2_train, 4),
                    'Test_R2': round(r2_test, 4),
                    'Train_RMSE': round(rmse_train, 4),
                    'Test_RMSE': round(rmse_test, 4)
                })

            # 📈 PDP 提取（Top N 变量）
            sorted_idx = np.argsort(best_model.feature_importances_)[::-1]
            top_features = [explanatory_vars[i] for i in sorted_idx[:9]]

            for feature in top_features:
                # 1) 新建一个仅用来提取 PDP 数据的 fig/ax，然后马上关闭
                fig, ax = plt.subplots()
                disp = PartialDependenceDisplay.from_estimator(best_model, X, [feature], ax=ax) # 使用X数据
                x_vals = disp.lines_[0][0].get_xdata()
                y_vals = disp.lines_[0][0].get_ydata()
                plt.close(fig)
                pdp_records.append({
                    'Feature': feature,
                    'GridSize': grid_size,
                    'Target': target,
                    'X': x_vals,
                    'Y': y_vals
                })

# 保存模型训练后的结果
df_all = pd.DataFrame(all_results)
df_all.to_excel(os.path.join(grid_folder, 'GBDT_Random_Search_Results.xlsx'), index=False)
df_r2 = pd.DataFrame(r2_comparison)
df_r2 = df_r2.sort_values(['Target', 'GridSize'])
df_r2.to_excel(os.path.join(grid_folder, 'R2_Comparison_Train_vs_Test.xlsx'), index=False)
print("✅ R² train vs test comparison saved.")
# 保存 PDP 数据
pdp_df = pd.DataFrame(pdp_records)
pdp_df.to_pickle(os.path.join(grid_folder, 'pdp_records.pkl'))  # 用 pickle 保留 numpy 数组

{np.int64(8), np.int64(5), np.int64(6), np.int64(7)}


In [8]:
import numpy as np
from scipy.stats import uniform

# 定义分布：起点为 0.545，跨度为 0.958
dist = uniform(loc=0.545, scale=0.958)

# 生成 10 万个样本
samples = dist.rvs(size=100000)

# 打印样本的最小值和最大值
print(f"min: {samples.min():.6f}")
print(f"max: {samples.max():.6f}")


{np.float64(0.7155410730108995), np.float64(0.6838977616293436), np.float64(0.8044988186283888), np.float64(0.8247625354150561), np.float64(0.9357003732473022), np.float64(0.741245447064351), np.float64(0.7965107408346093), np.float64(0.636515935475245), np.float64(0.9402259373884085), np.float64(0.9305707544359061), np.float64(0.9328854533041806), np.float64(0.7211858930295528), np.float64(0.6532341873127728), np.float64(0.6525380664462653), np.float64(0.557538465172762), np.float64(0.7305726365355769), np.float64(0.8472691975958571), np.float64(0.5668977120805646), np.float64(0.9409577021371017), np.float64(0.6352558711480332), np.float64(0.8463770837244202), np.float64(0.7480837914393281), np.float64(0.644782598611457), np.float64(0.7290645847728255), np.float64(0.8464028456644966), np.float64(0.5940027883001987), np.float64(0.5457600680358295), np.float64(0.7094827485611368), np.float64(0.8952608661450163), np.float64(0.8818766557595914), np.float64(0.6925029085441751), np.float64(

In [16]:
import random
# 生成一个在1到10之间的随机整数
random_number = random.randint(1, 10)
print(random_number)

7


In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# === 文件路径 ===
grid_folder = r'D:\seoul\grids\lst_map'
input_path = os.path.join(grid_folder, 'R2_Comparison_Train_vs_Test.xlsx')
output_dir = os.path.join(grid_folder, 'figures', 'r2_comparison')
os.makedirs(output_dir, exist_ok=True)

# === 读取数据 ===
df = pd.read_excel(input_path)
df['GridSize'] = df['GridSize'].astype(int)

# === 绘图：每个 Target 一个对比图 ===
for target in df['Target'].unique():
    df_sub = df[df['Target'] == target].sort_values('GridSize')

    plt.figure(figsize=(8, 5))
    plt.plot(df_sub['GridSize'], df_sub['Train_R2'], marker='o', label='Train R²')
    plt.plot(df_sub['GridSize'], df_sub['Test_R2'], marker='o', label='Test R²')

    plt.title(f'R² Comparison — {target}')
    plt.xticks(sorted(df['GridSize'].unique()))  # ✅ 强制显示真实 X 值
    plt.xlabel('Grid Size (m)')
    plt.ylabel('R² Score')
    plt.ylim(0, 1.05)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    out_path = os.path.join(output_dir, f'R2_Train_vs_Test_{target}.png')
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"📈 Saved R² comparison: {out_path}")


📈 Saved R² comparison: D:\seoul\grids\lst_map\figures\r2_comparison\R2_Train_vs_Test_ext_2020.png
📈 Saved R² comparison: D:\seoul\grids\lst_map\figures\r2_comparison\R2_Train_vs_Test_hr_2020.png
📈 Saved R² comparison: D:\seoul\grids\lst_map\figures\r2_comparison\R2_Train_vs_Test_nor_2020.png


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os

# 路径
grid_folder = r'D:\seoul\grids\lst_map'
fig_output_dir = os.path.join(grid_folder, 'figures', 'pdp_gridsearch_combined')
os.makedirs(fig_output_dir, exist_ok=True)

# 读取 PDP 数据
pdp_df = pd.read_pickle(os.path.join(grid_folder, 'pdp_records.pkl'))  # 之前保存的 pickle

# 画图
for (feature, target), group in pdp_df.groupby(['Feature', 'Target']):
    plt.figure(figsize=(7, 8))
    group_sorted = group.copy()
    group_sorted['GridSize'] = group_sorted['GridSize'].astype(int)
    group_sorted = group_sorted.sort_values('GridSize')
    colors = cm.viridis(np.linspace(0, 1, len(group_sorted)))

    for idx, (_, row) in enumerate(group_sorted.iterrows()):
        gridsize = row['GridSize']
        if gridsize == 450:  # 就是450
            plt.plot(row['X'], row['Y'], label=f"{gridsize}m (baseline)", color='red', linewidth=3, linestyle='-')
        else:
            plt.plot(row['X'], row['Y'], label=f"{gridsize}m", color=colors[idx], linewidth=2)

    plt.title(f"{feature} — {target}")
    plt.xlabel(feature)
    plt.ylabel(f"Partial dependence on {target}")
    plt.legend(title='Grid Size')
    plt.tight_layout()
    plt.grid(True)

    fig_path = os.path.join(fig_output_dir, f'pdp_{target}_{feature}_allgrids.png')
    plt.savefig(fig_path, dpi=300)
    plt.close()
    print(f"📊 Saved PDP: {fig_path}")

📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_ext_2020_BCR(%)_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_hr_2020_BCR(%)_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_nor_2020_BCR(%)_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_ext_2020_BHV_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_hr_2020_BHV_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_nor_2020_BHV_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_ext_2020_Dist_BP_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_hr_2020_Dist_BP_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_nor_2020_Dist_BP_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_ext_2020_Dist_MT_allgrids.pn

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 📁 路径设置
grid_folder = r'D:\seoul\grids\lst_map'
input_path = os.path.join(grid_folder, 'GBDT_GridSearch_Results.xlsx')
output_dir = os.path.join(grid_folder, 'figures', 'grid_comparison')
os.makedirs(output_dir, exist_ok=True)

# 📄 读取数据
df = pd.read_excel(input_path)
df['GridSize'] = df['GridSize'].astype(int)

# ✅ 提取每个 GridSize + Target 的最佳参数（去重）
best_params = df.drop_duplicates(subset=['GridSize', 'Target'])[
    ['GridSize', 'Target', 'learning_rate', 'max_depth', 'n_estimators', 'subsample']
].sort_values(['Target', 'GridSize'])

# 💾 保存为单独 Excel
param_path = os.path.join(grid_folder, 'GBDT_BestParams_byGrid.xlsx')
best_params.to_excel(param_path, index=False)
print(f"✅ 已保存最优参数表：{param_path}")

# 📈 R² 和 RMSE 折线图
for metric in ['R2', 'RMSE']:
    plt.figure(figsize=(8, 5))
    for target in df['Target'].unique():
        sub_df = df[df['Target'] == target]
        sub_df = sub_df.groupby('GridSize')[metric].mean().reset_index()

        # 👉 取这个目标变量的参数示例（第一个 GridSize）
        example_param = best_params[best_params['Target'] == target].iloc[0]
        param_text = f"lr={example_param['learning_rate']}, depth={example_param['max_depth']}, est={example_param['n_estimators']}"

        plt.plot(sub_df['GridSize'], sub_df[metric], marker='o', label=f"{target} ({param_text})")

    plt.title(f'{metric} vs Grid Size (Best GBDT via GridSearchCV)')
    plt.xlabel('Grid Size (m)')
    plt.ylabel(metric)
    plt.grid(True)
    plt.legend(title='Target + Params', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{metric}_Line_by_GridSize_withParams.png'), dpi=300)
    plt.close()

print(f"✅ 折线图已保存至：{output_dir}")


# 🔥 2. 特征重要性热力图（每个变量 × grid size，按平均值）
pivot = df.pivot_table(index='Feature', columns='GridSize', values='Importance', aggfunc='mean')
plt.figure(figsize=(12, 6))
sns.heatmap(pivot, cmap='YlOrBr', annot=True, fmt=".2f")
plt.title('Feature Importance (Mean) by Grid Size')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'FeatureImportance_Heatmap_by_Grid.png'), dpi=300)
plt.close()

# 📈 3. 每个 Feature 的重要性随 GridSize 变化线图（按目标变量分别画）
for target in df['Target'].unique():
    df_target = df[df['Target'] == target]
    grouped = df_target[['GridSize', 'Feature', 'Importance']].copy()

    plt.figure(figsize=(5, 6))
    sns.lineplot(data=grouped, x='GridSize', y='Importance', hue='Feature', marker='o')

    plt.title(f'Feature Importance per Grid Size — {target}')
    plt.xlabel('Grid Size (m)')
    plt.ylabel('Feature Importance')
    plt.xticks(sorted(df_target['GridSize'].unique()))  # ✅ 强制显示真实 X 值
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'ImportanceTrend_{target}.png'), dpi=300)
    plt.close()


print(f"✅ 所有按 Grid 对比图保存至：{output_dir}")


✅ 已保存最优参数表：D:\seoul\grids\lst_map\GBDT_BestParams_byGrid.xlsx
✅ 折线图已保存至：D:\seoul\grids\lst_map\figures\grid_comparison
✅ 所有按 Grid 对比图保存至：D:\seoul\grids\lst_map\figures\grid_comparison


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

param_cols = ['learning_rate', 'max_depth', 'n_estimators', 'subsample']
df_params = df.drop_duplicates(subset=['GridSize', 'Target'] + param_cols).copy()
df_params['GridSize'] = df_params['GridSize'].astype(int)
df_params = df_params.sort_values(['Target', 'GridSize'])

# 创建输出路径
output_dir = os.path.join(grid_folder, 'figures', 'hyperparam_analysis')
os.makedirs(output_dir, exist_ok=True)

# === 折线图：连续型参数趋势 ===
for param in ['learning_rate', 'subsample']:
    plt.figure(figsize=(8, 5))
    for target in df_params['Target'].unique():
        df_sub = df_params[df_params['Target'] == target]
        plt.plot(df_sub['GridSize'], df_sub[param], marker='o', label=target)

    plt.title(f'{param} across Grid Sizes')
    plt.xlabel('Grid Size (m)')
    plt.ylabel(param)
    plt.grid(True)
    plt.legend(title='Target')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{param}_trend.png'), dpi=300)
    plt.close()

# === 柱状图：分类型参数频率 ===
for param in ['max_depth', 'n_estimators']:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df_params, x='GridSize', hue=param, palette='Set2')
    plt.title(f'{param} distribution across Grid Sizes')
    plt.xlabel('Grid Size (m)')
    plt.ylabel('Count')
    plt.legend(title=param)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{param}_count.png'), dpi=300)
    plt.close()

print(f"✅ 所有超参数比较图已保存到：{output_dir}")


✅ 所有超参数比较图已保存到：D:\seoul\grids\lst_map\figures\hyperparam_analysis


In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 读取用户上传的 Excel 文件
input_path = "D:\seoul\grids\lst_map/GBDT_GridSearch_Results.xlsx"
df = pd.read_excel(input_path)

# 转换 GridSize 为整数型，便于排序和绘图
df['GridSize'] = df['GridSize'].astype(int)

# 创建输出文件夹
output_dir = "/mnt/data/gridsearch_param_visualization"
os.makedirs(output_dir, exist_ok=True)

# 筛选唯一的最优参数组合（按 GridSize 和 Target）
best_params = df.drop_duplicates(subset=['GridSize', 'Target'])[
    ['GridSize', 'Target', 'learning_rate', 'max_depth', 'n_estimators', 'subsample']
].sort_values(['Target', 'GridSize'])

# 绘制每个超参数随 GridSize 变化的线图，按目标变量分图
param_list = ['learning_rate', 'max_depth', 'n_estimators', 'subsample']
for param in param_list:
    for target in best_params['Target'].unique():
        subset = best_params[best_params['Target'] == target]
        plt.figure(figsize=(8, 5))
        sns.lineplot(data=subset, x='GridSize', y=param, marker='o')
        plt.title(f"{param} vs GridSize — {target}")
        plt.xlabel("Grid Size (m)")
        plt.ylabel(param)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{param}_vs_GridSize_{target}.png"), dpi=300)
        plt.close()

import ace_tools as tools; tools.display_dataframe_to_user(name="Best GBDT Parameters", dataframe=best_params)


<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
C:\Users\owner\AppData\Local\Temp\ipykernel_27964\3670598890.py:7: SyntaxWarning: invalid escape sequence '\s'
  input_path = "D:\seoul\grids\lst_map/GBDT_GridSearch_Results.xlsx"
C:\Users\owner\AppData\Local\Temp\ipykernel_27964\3670598890.py:7: SyntaxWarning: invalid escape sequence '\s'
  input_path = "D:\seoul\grids\lst_map/GBDT_GridSearch_Results.xlsx"


ModuleNotFoundError: No module named 'ace_tools'

# 02 regression model

## 01 Initial Modeling: Ordinary least squares (OLS) regression models.

首先使用普通最小二乘法（Ordinary Least Squares, OLS）建立基础回归模型。
评估模型解释力和残差行为。

In [22]:
import os
import geopandas as gpd
import statsmodels.api as sm
import re

# 📁 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map'

# 🔧 变量定义
target_vars = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']

# === 主循环 ===
for filename in os.listdir(grid_folder):
    if filename.endswith('_clean.shp'):
        filepath = os.path.join(grid_folder, filename)
        print(f"Processing {filename}...")

        # 读取 shapefile
        gdf = gpd.read_file(filepath)

        for target in target_vars:
            y = gdf[target]
            X = gdf[explanatory_vars]
            '''
            y = β₁X₁ + β₂X₂ + ...
            '''
            X = sm.add_constant(X)  # 添加截距项
            ''' -> 截距的意思是
            y = β₁X₁ + β₂X₂ + ...+ ε
            '''

            model = sm.OLS(y, X)
            results = model.fit()

            # 提取文件名中的 3~5 位数字
            match = re.search(r'grid_(\d{3,5})m', filename)
            file_id = match.group() if match else "Unknown"

            print(f"OLS results for {file_id} | target: {target}")
            #print(results.summary())

            print(f"R²: {results.rsquared:.4f}")
            rmse = np.sqrt(np.mean(results.resid ** 2))
            print(f"RMSE: {rmse:.4f}")


Processing city2020_lst_ratio_grid_1080m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...
OLS results for grid_1080m | target: nor_2020
R²: 0.8555
RMSE: 1.1429
OLS results for grid_1080m | target: ext_2020
R²: 0.9165
RMSE: 1.1774
OLS results for grid_1080m | target: hr_2020
R²: 0.6943
RMSE: 0.7749
Processing city2020_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...
OLS results for grid_120m | target: nor_2020
R²: 0.7772
RMSE: 1.6511
OLS results for grid_120m | target: ext_2020
R²: 0.8458
RMSE: 1.8562
OLS results for grid_120m | target: hr_2020
R²: 0.6493
RMSE: 0.9616
Processing city2020_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...
OLS results for grid_240m | target: nor_2020
R²: 0.8145
RMSE: 1.4714
OLS results for grid_240m | target: ext_2020
R²: 0.8775
RMSE: 1.6247
OLS results for grid_240m | target: hr_2020
R²: 0.6752
RMSE: 0.9078
Processing city2020_lst_ratio_grid_360m_bcr_bhv_ndvi_svf_ev_distbp_distmt_dist

## 02 Test for Spatial Autocorrelation: Moran’s I statistics

使用 Moran’s I 统计量对 OLS 残差进行空间自相关检验。
若结果显著，说明存在空间自相关，OLS 模型不适用，需考虑空间回归模型。

In [24]:
import geopandas as gpd
import numpy as np
from esda.moran import Moran
from libpysal.weights import Queen
import matplotlib.pyplot as plt

for filename in os.listdir(grid_folder):
    if filename.endswith('_clean.shp'):
        filepath = os.path.join(grid_folder, filename)
        print(f"Processing {filename}...")

        # 读取 shapefile
        gdf = gpd.read_file(filepath)

        # 2. 选择分析的目标变量（假设叫 'target_var'）
        y = gdf[target]
        X = gdf[explanatory_vars]

        # 3. 构建空间权重矩阵，Queen 邻接权重
        w = Queen.from_dataframe(gdf, use_index=False)
        w.transform = 'r'  # 行标准化

        # 4. 计算Moran's I
        moran = Moran(y, w)

        # 5. 输出结果
        print(f"Moran's I: {moran.I:.4f}")
        print(f"p-value: {moran.p_sim:.4f}")  # 基于随机化的显著性检验p值
        print(f"Expected I: {moran.EI:.4f}")
        print(f"Variance: {moran.VI_sim:.4f}")

        # 6. 判断空间自相关是否显著
        if moran.p_sim < 0.05:
            print("Significant spatial autocorrelation detected.")
        else:
            print("No significant spatial autocorrelation detected.")

Processing city2020_lst_ratio_grid_1080m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...
Moran's I: 0.6315
p-value: 0.0010
Expected I: -0.0016
Variance: 0.0005
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Moran's I: 0.9266
p-value: 0.0010
Expected I: -0.0000
Variance: 0.0000
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
 There is 1 island with id: 367.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 367, ' is an island (no neighbors)')
Moran's I: 0.8587
p-value: 0.0010
Expected I: -0.0001
Variance: 0.0000
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_360m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...
Moran's I: 0.8061
p-value: 0.0010
Expected I: -0.0002
Variance: 0.0001
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_450m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...


...

('WARNING: ', 111, ' is an island (no neighbors)')
Moran's I: 0.7720
p-value: 0.0010
Expected I: -0.0003
Variance: 0.0001
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
 There is 1 island with id: 94.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 94, ' is an island (no neighbors)')
Moran's I: 0.7597
p-value: 0.0010
Expected I: -0.0004
Variance: 0.0001
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_600m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
 There is 1 island with id: 60.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 60, ' is an island (no neighbors)')
Moran's I: 0.7229
p-value: 0.0010
Expected I: -0.0005
Variance: 0.0002
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_720m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...
Moran's I: 0.6845
p-value: 0.0010
Expected I: -0.0008
Variance: 0.0002
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_840m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...
Moran's I: 0.6504
p-value: 0.0010
Expected I: -0.0010
Variance: 0.0003
Significant spatial autocorrelation detected.
Processing city2020_lst_ratio_grid_960m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp...
Moran's I: 0.6427
p-value: 0.0010
Expected I: -0.0013
Variance: 0.0004
Significant spatial autocorrelation detected.


Moran's I = 0.6315
表示变量存在较强的正空间自相关，也就是说，空间上相邻的区域的取值更趋于相似。
p-value = 0.0010
通过随机化检验得到的p值非常小（远小于0.05），说明这种空间自相关是显著的，不是随机出现的。
Expected I = -0.0016
这是在空间随机分布下，Moran’s I的期望值，接近0，表明无空间自相关时的基准。
Variance = 0.0005
表示Moran's I的估计方差，用于计算统计显著性。
结论
数据有显著的空间聚集效应，空间上相似的值在地理位置上是聚集的。

## 03 Identify the Type of Spatial Dependence: Lagrange Multiplier statistics
使用 Lagrange Multiplier（LM）检验 判断空间依赖类型：
* LM-Lag（滞后型）：是否存在因变量的空间滞后项。
* LM-Error（误差型）：是否存在误差项的空间相关性。
* 同时参考 Robust LM-Lag 和 Robust LM-Error（控制另一种依赖类型后的检验）。
* 如果两个 Robust LM 检验都显著，说明两种依赖都可能存在，建议使用更一般形式的模型，如 SDM。

In [ ]:
!{sys.executable} -m pip install -U spreg

In [34]:
# 1. 安装依赖（若尚未安装）
# pip install pysal geopandas spreg

# 2. 导入必要的库
import os
import numpy as np
import geopandas as gpd
import libpysal
from spreg import OLS, LMtests     # 正确的导入方式

# 3. 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map'

# 4. 变量定义
target_vars = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = [
    'BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)'
]

# === 主循环：遍历每个 *_clean.shp 并对每个 target 执行 LM 检验 ===
for filename in os.listdir(grid_folder):
    if not filename.endswith('_clean.shp'):
        continue

    filepath = os.path.join(grid_folder, filename)
    print(f"\nProcessing file: {filename}")

    # 5. 读取 GeoDataFrame
    gdf = gpd.read_file(filepath)

    # 6. 构建空间权重（Queen 邻接，行标准化）
    w = libpysal.weights.Queen.from_dataframe(gdf)
    w.transform = 'r'

    # 7. 对每个因变量做诊断
    for target in target_vars:
        print(f"\n→ Target variable: {target}")

        # 7.1 构造 y 和 X
        y = gdf[target].values.reshape(-1, 1)
        X = gdf[explanatory_vars].values
        # 如果想自动添加截距，可以改为：
        # X = np.hstack([np.ones((len(gdf),1)), X])

        # 7.2 运行 OLS
        ols_model = OLS(y, X, name_y=target, name_x=explanatory_vars)

        # 7.3 运行 LM 系列检验
        lms = LMtests(ols_model, w)

        # 7.4 输出结果（注意属性名：lml, lme, rlml, rlme）
        print(f"  LM-Lag (lml)                   : {lms.lml[0]:.4f}, p-value = {lms.lml[1]:.4g}")
        print(f"  LM-Error (lme)                 : {lms.lme[0]:.4f}, p-value = {lms.lme[1]:.4g}")
        print(f"  Robust LM-Lag (rlml)           : {lms.rlml[0]:.4f}, p-value = {lms.rlml[1]:.4g}")
        print(f"  Robust LM-Error (rlme)         : {lms.rlme[0]:.4f}, p-value = {lms.rlme[1]:.4g}")

        # 7.5 判断建议
        #    - 若 lml 或 rlml 显著（e.g., p < 0.05），说明空间滞后依赖；
        #    - 若 lme 或 rlme 显著，说明空间误差依赖；
        #    - 若两种 Robust 检验都显著，则建议考虑更通用的 SDM 模型。



Processing file: city2020_lst_ratio_grid_1080m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp

→ Target variable: nor_2020
  LM-Lag (lml)                   : 311.0677, p-value = 1.278e-69
  LM-Error (lme)                 : 787.1751, p-value = 3.315e-173
  Robust LM-Lag (rlml)           : 9.2159, p-value = 0.002399
  Robust LM-Error (rlme)         : 485.3234, p-value = 1.484e-107

→ Target variable: ext_2020
  LM-Lag (lml)                   : 156.8978, p-value = 5.389e-36
  LM-Error (lme)                 : 484.8838, p-value = 1.849e-107
  Robust LM-Lag (rlml)           : 11.9506, p-value = 0.0005463
  Robust LM-Error (rlme)         : 339.9366, p-value = 6.587e-76

→ Target variable: hr_2020
  LM-Lag (lml)                   : 540.9465, p-value = 1.174e-119
  LM-Error (lme)                 : 876.9513, p-value = 1.006e-192
  Robust LM-Lag (rlml)           : 8.1639, p-value = 0.004273
  Robust LM-Error (rlme)         : 344.1687, p-value = 7.889e-77

Processing file: city2020_lst_

C:\Users\owner\AppData\Local\Temp\ipykernel_54380\1020691830.py:33: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf)
C:\Users\owner\AppData\Local\Temp\ipykernel_54380\1020691830.py:33: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)



→ Target variable: nor_2020
  LM-Lag (lml)                   : 71897.0810, p-value = 0
  LM-Error (lme)                 : 87154.4945, p-value = 0
  Robust LM-Lag (rlml)           : 6103.9043, p-value = 0
  Robust LM-Error (rlme)         : 21361.3178, p-value = 0

→ Target variable: ext_2020
  LM-Lag (lml)                   : 62196.0344, p-value = 0
  LM-Error (lme)                 : 74385.6309, p-value = 0
  Robust LM-Lag (rlml)           : 8553.9648, p-value = 0
  Robust LM-Error (rlme)         : 20743.5613, p-value = 0

→ Target variable: hr_2020
  LM-Lag (lml)                   : 97014.8009, p-value = 0
  LM-Error (lme)                 : 112452.3287, p-value = 0
  Robust LM-Lag (rlml)           : 2470.5138, p-value = 0
  Robust LM-Error (rlme)         : 17908.0415, p-value = 0

Processing file: city2020_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp


C:\Users\owner\AppData\Local\Temp\ipykernel_54380\1020691830.py:33: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
 There is 1 island with id: 367.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 367, ' is an island (no neighbors)')

→ Target variable: nor_2020
  LM-Lag (lml)                   : 11545.9786, p-value = 0
  LM-Error (lme)                 : 19129.0784, p-value = 0
  Robust LM-Lag (rlml)           : 683.5475, p-value = 1.131e-150
  Robust LM-Error (rlme)         : 8266.6473, p-value = 0

→ Target variable: ext_2020
  LM-Lag (lml)                   : 9111.3179, p-value = 0
  LM-Error (lme)                 : 15620.9937, p-value = 0
  Robust LM-Lag (rlml)           : 992.5713, p-value = 7.396e-218
  Robust LM-Error (rlme)         : 7502.2471, p-value = 0

→ Target variable: hr_2020
  LM-Lag (lml)                   : 18424.3445, p-value = 0
  LM-Error (lme)                 : 24577.0885, p-value = 0
  Robust LM-Lag (rlml)           : 284.4121, p-value = 8.206e-64
  Robust LM-Error (rlme)         : 6437.1562, p-value = 0

Processing file: city2020_lst_ratio_grid_360m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp


...

C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
 There is 1 island with id: 60.
  W.__init__(self, neighbors, ids=ids, **kw)
C:\Users\owner\AppData\Local\Temp\ipykernel_54380\1020691830.py:33: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf)


('WARNING: ', 60, ' is an island (no neighbors)')

→ Target variable: nor_2020
  LM-Lag (lml)                   : 902.3419, p-value = 3.039e-198
  LM-Error (lme)                 : 2581.7374, p-value = 0
  Robust LM-Lag (rlml)           : 36.8627, p-value = 1.267e-09
  Robust LM-Error (rlme)         : 1716.2583, p-value = 0

→ Target variable: ext_2020
  LM-Lag (lml)                   : 601.9108, p-value = 6.43e-133
  LM-Error (lme)                 : 1712.3942, p-value = 0
  Robust LM-Lag (rlml)           : 65.1844, p-value = 6.821e-16
  Robust LM-Error (rlme)         : 1175.6678, p-value = 1.184e-257

→ Target variable: hr_2020
  LM-Lag (lml)                   : 1896.2496, p-value = 0
  LM-Error (lme)                 : 3163.5523, p-value = 0
  Robust LM-Lag (rlml)           : 25.2518, p-value = 5.031e-07
  Robust LM-Error (rlme)         : 1292.5545, p-value = 4.691e-283

Processing file: city2020_lst_ratio_grid_720m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp

→ Target var

C:\Users\owner\AppData\Local\Temp\ipykernel_54380\1020691830.py:33: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf)


  LM-Lag (lml)                   : 563.0869, p-value = 1.791e-124
  LM-Error (lme)                 : 1321.2716, p-value = 2.695e-289
  Robust LM-Lag (rlml)           : 23.5983, p-value = 1.187e-06
  Robust LM-Error (rlme)         : 781.7830, p-value = 4.93e-172

→ Target variable: ext_2020
  LM-Lag (lml)                   : 293.9458, p-value = 6.868e-66
  LM-Error (lme)                 : 786.6239, p-value = 4.368e-173
  Robust LM-Lag (rlml)           : 28.5602, p-value = 9.083e-08
  Robust LM-Error (rlme)         : 521.2383, p-value = 2.276e-115

→ Target variable: hr_2020
  LM-Lag (lml)                   : 829.5852, p-value = 1.995e-182
  LM-Error (lme)                 : 1358.1372, p-value = 2.627e-297
  Robust LM-Lag (rlml)           : 12.8158, p-value = 0.0003437
  Robust LM-Error (rlme)         : 541.3677, p-value = 9.504e-120

Processing file: city2020_lst_ratio_grid_960m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp

→ Target variable: nor_2020
  LM-Lag (lml)          

C:\Users\owner\AppData\Local\Temp\ipykernel_54380\1020691830.py:33: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf)


## 04 Model Comparison: Likelihood Ratio (LR) Tests
在模型之间进行似然比检验（Likelihood Ratio Test），判断引入空间项后模型改进是否显著。
比较：
OLS vs SEM / SLM / SDM
SLM vs SDM / SEM vs SDM

In [ ]:
# 0. 安装依赖（若尚未安装）
# pip install pysal geopandas scipy spreg

# 1. 导入必要库
import os
import numpy as np
from scipy.stats import chi2
import geopandas as gpd
import libpysal
from spreg import OLS, ML_Error, ML_Lag  # 不再 import SDM

# 2. 文件夹路径与变量定义
grid_folder      = r'D:\seoul\grids\lst_map'
target_vars      = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = [
    'BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)'
]

# 3. 主循环：遍历每个 *_clean.shp 并对每个 target 执行 LR 测试
for filename in os.listdir(grid_folder):
    if not filename.endswith('_clean.shp'):
        continue

    path = os.path.join(grid_folder, filename)
    print(f"\nProcessing {filename}")
    gdf = gpd.read_file(path)

    # 构造 y, X, w
    y = gdf[target_vars].values  # shape (n, 3)
    # 这里我们对每个因变量分别跑模型，因此循环内部再拆
    w = libpysal.weights.Queen.from_dataframe(gdf)
    w.transform = 'r'

    for i, target in enumerate(target_vars):
        yi = y[:, i].reshape(-1, 1)
        X0 = gdf[explanatory_vars].values
        X  = np.hstack([np.ones((len(gdf),1)), X0])
        name_x = ['const'] + explanatory_vars

        # 4. 依次估计四个模型
        models = {}
        models['OLS'] = OLS( yi, X, name_y=target, name_x=name_x )
        models['SEM'] = ML_Error( yi, X, w=w,       name_y=target, name_x=name_x )
        models['SLM'] = ML_Lag(  yi, X, w=w,       name_y=target, name_x=name_x )
        # SDM 用 ML_Lag + slx_lags=1
        models['SDM'] = ML_Lag(  yi, X, w=w, slx_lags=1, name_y=target, name_x=name_x )

        # 5. 提取对数似然
        ll = {name: mdl.logll for name, mdl in models.items()}

        # 6. LR 检验函数
        def lr_test(ll_r, ll_f, df_diff):
            lr = 2 * (ll_f - ll_r)
            p  = chi2.sf(lr, df_diff)
            return lr, p

        # 7. 设定自由度差
        p = X.shape[1] - 1  # 排除常数
        tests = [
            ('OLS','SEM', 1),
            ('OLS','SLM', 1),
            ('OLS','SDM', p+1),  # SDM 多了 rho + p 个 theta
            ('SLM','SDM',   p),
            ('SEM','SDM',   p),
        ]

        # 8. 输出每对检验
        print(f"\n → Dependent = {target}")
        for r, f, df_diff in tests:
            stat, pval = lr_test(ll[r], ll[f], df_diff)
            print(f"   {r:>4} vs {f:<4} | χ²({df_diff}) = {stat:.2f}, p-value = {pval:.3g}")



Processing city2020_lst_ratio_grid_1080m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp


C:\Users\owner\AppData\Local\Temp\ipykernel_54380\1177523121.py:32: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(



 → Dependent = nor_2020
    OLS vs SEM  | χ²(1) = 527.91, p-value = 8.06e-117
    OLS vs SLM  | χ²(1) = 273.16, p-value = 2.33e-61
    OLS vs SDM  | χ²(10) = 536.99, p-value = 5.45e-109
    SLM vs SDM  | χ²(9) = 263.83, p-value = 1.19e-51
    SEM vs SDM  | χ²(9) = 9.08, p-value = 0.43


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(



 → Dependent = ext_2020
    OLS vs SEM  | χ²(1) = 324.30, p-value = 1.67e-72
    OLS vs SLM  | χ²(1) = 146.66, p-value = 9.32e-34
    OLS vs SDM  | χ²(10) = 340.89, p-value = 3.42e-67
    SLM vs SDM  | χ²(9) = 194.23, p-value = 5.36e-37
    SEM vs SDM  | χ²(9) = 16.58, p-value = 0.0556


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(



 → Dependent = hr_2020
    OLS vs SEM  | χ²(1) = 609.08, p-value = 1.77e-134
    OLS vs SLM  | χ²(1) = 419.75, p-value = 2.77e-93
    OLS vs SDM  | χ²(10) = 634.89, p-value = 5.86e-130
    SLM vs SDM  | χ²(9) = 215.14, p-value = 2.2e-41
    SEM vs SDM  | χ²(9) = 25.81, p-value = 0.0022

Processing city2020_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp


C:\Users\owner\AppData\Local\Temp\ipykernel_54380\1177523121.py:32: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:563: RuntimeWarning: divide by zero encountered in log
  jacob = np.log(np.linalg.det(a))


## 05 Model Selection Based on Diagnostics
选择合适的空间回归模型
检验结果	推荐模型
仅 LM-Error 显著	SEM（空间误差模型）
仅 LM-Lag 显著	SLM（空间滞后模型）
二者都显著	SDM（空间Durbin模型） 或进行进一步LR检验对比
SDM 是最一般的模型，可以退化为 SLM 或 SEM。
若 SDM 中某些系数不显著，可考虑退化为简化模型。

## 06 Final Diagnostics and Interpretation
检查模型残差分布、异方差性、解释变量的空间效应（直接效应、间接效应）。
分析结果的空间意义与实际城市空间结构/环境变量的关系。

In [1]:
# 安装依赖（如未安装）
# pip install pysal geopandas pandas spreg scipy

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from spreg import ML_Lag
from scipy.stats import norm

# 支持 libpysal 和 pysal.lib 两种导入
try:
    import libpysal
    Queen = libpysal.weights.Queen
except ImportError:
    from pysal.lib.weights import Queen

# 参数
grid_folder      = r'D:\seoul\grids\lst_map'
filename         = 'city2020_lst_ratio_grid_450m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp'
path             = os.path.join(grid_folder, filename)

target_vars      = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = [
    'BCR(%)','BHV','NDVI','SVF','EV(m)',
    'Dist_BP','Dist_MT','Dist_WB','WR(%)'
]

# 读取数据 & 构造空间权重
gdf = gpd.read_file(path)
w   = Queen.from_dataframe(gdf)
w.transform = 'r'
n   = len(gdf)

# 准备结果表
coef_df  = pd.DataFrame(index=target_vars, columns=explanatory_vars, dtype=float)
pval_df  = coef_df.copy()
lag_coef = coef_df.copy()
lag_pval = coef_df.copy()

# 循环估计 SDM 并收集系数与 p-value
for target in target_vars:
    y  = gdf[target].values.reshape(-1,1)
    X0 = gdf[explanatory_vars].values
    X  = np.hstack([np.ones((n,1)), X0])  # add constant

    model = ML_Lag(y, X, w=w, slx_lags=1)
    betas = model.betas.flatten()   # [const, βs..., θs..., rho]
    stats = model.z_stat            # [(z, p-value), …]

    # 直接系数 β 和 p-value
    for i, var in enumerate(explanatory_vars):
        coef_df.loc[target, var]  = betas[i+1]
        pval_df.loc[target, var]  = stats[i+1][1]

    # 空间滞后系数 θ 和 p-value
    offset = 1 + len(explanatory_vars)
    for i, var in enumerate(explanatory_vars):
        lag_coef.loc[target, var] = betas[offset + i]
        lag_pval.loc[target, var] = stats[offset + i][1]

# 用小数点后四位格式化 p-value
pval_fmt     = pval_df.applymap(lambda p: f"{p:.4f}")
lag_pval_fmt = lag_pval.applymap(lambda p: f"{p:.4f}")

coef_table = coef_df.round(4).astype(str)  + " (" + pval_fmt  + ")"
lag_table  = lag_coef.round(4).astype(str) + " (" + lag_pval_fmt + ")"

# 重命名滞后变量列
lag_table = lag_table.rename(columns={col: f"lagged_{col}" for col in lag_table.columns})

# 输出
print("\nRegression Coefficients and Significance:")
print(coef_table)
print("\nSpatial Lag Regression Results (Lagged Variables):")
print(lag_table)

C:\Users\owner\AppData\Local\Temp\ipykernel_2600\3906064511.py:31: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w   = Queen.from_dataframe(gdf)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
 There is 1 island with id: 111.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 111, ' is an island (no neighbors)')

Regression Coefficients and Significance:
                    BCR(%)              BHV               NDVI  \
nor_2020   0.0522 (0.0000)  0.0037 (0.3256)   -17.076 (0.0000)   
ext_2020   0.0572 (0.0000)  -0.004 (0.3762)  -23.1788 (0.0000)   
hr_2020   -0.0074 (0.0000)  0.0053 (0.0107)    5.9465 (0.0000)   

                      SVF             EV(m)           Dist_BP  \
nor_2020  8.6633 (0.0000)  -0.0093 (0.0000)   0.0003 (0.0017)   
ext_2020  8.2828 (0.0000)  -0.0125 (0.0000)   0.0004 (0.0053)   
hr_2020   0.1303 (0.5368)   0.0032 (0.0000)  -0.0001 (0.0174)   

                   Dist_MT           Dist_WB             WR(%)  
nor_2020   0.0017 (0.0000)      0.0 (0.7814)  -0.1206 (0.0000)  
ext_2020   0.0018 (0.0000)  -0.0001 (0.6880)  -0.1745 (0.0000)  
hr_2020   -0.0003 (0.0000)      0.0 (0.5954)   0.0525 (0.0000)  

Spatial Lag Regression Results (Lagged Variables):
             lagged_BCR(%)        lagged_BHV       lagged_NDVI  \
no

C:\Users\owner\AppData\Local\Temp\ipykernel_2600\3906064511.py:63: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pval_fmt     = pval_df.applymap(lambda p: f"{p:.4f}")
C:\Users\owner\AppData\Local\Temp\ipykernel_2600\3906064511.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lag_pval_fmt = lag_pval.applymap(lambda p: f"{p:.4f}")


In [2]:
coef_table

,BCR(%),BHV,NDVI,SVF,EV(m),Dist_BP,Dist_MT,Dist_WB,WR(%)
nor_2020,0.0522 (0.0000),0.0037 (0.3256),-17.076 (0.0000),8.6633 (0.0000),-0.0093 (0.0000),0.0003 (0.0017),0.0017 (0.0000),0.0 (0.7814),-0.1206 (0.0000)
ext_2020,0.0572 (0.0000),-0.004 (0.3762),-23.1788 (0.0000),8.2828 (0.0000),-0.0125 (0.0000),0.0004 (0.0053),0.0018 (0.0000),-0.0001 (0.6880),-0.1745 (0.0000)
hr_2020,-0.0074 (0.0000),0.0053 (0.0107),5.9465 (0.0000),0.1303 (0.5368),0.0032 (0.0000),-0.0001 (0.0174),-0.0003 (0.0000),0.0 (0.5954),0.0525 (0.0000)


In [3]:
lag_table

,lagged_BCR(%),lagged_BHV,lagged_NDVI,lagged_SVF,lagged_EV(m),lagged_Dist_BP,lagged_Dist_MT,lagged_Dist_WB,lagged_WR(%)
nor_2020,-0.0589 (0.0000),-0.0319 (0.0000),10.9877 (0.0000),-8.7437 (0.0000),0.008 (0.0000),-0.0003 (0.0083),-0.0017 (0.0000),-0.0001 (0.6371),0.0899 (0.0000)
ext_2020,-0.07 (0.0000),-0.0379 (0.0000),13.8717 (0.0000),-8.8813 (0.0000),0.0099 (0.0000),-0.0003 (0.0166),-0.0018 (0.0000),0.0 (0.9235),0.1212 (0.0000)
hr_2020,0.0167 (0.0000),0.0118 (0.0037),-4.4519 (0.0000),1.3184 (0.0005),-0.0024 (0.0000),0.0002 (0.0175),0.0003 (0.0000),-0.0 (0.7359),-0.0419 (0.0000)


In [ ]:
# 安装依赖（如未安装）
# pip install pysal geopandas pandas spreg scipy scikit-learn

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import libpysal
from spreg import ML_Lag
from sklearn.metrics import mean_squared_error, mean_absolute_error


# 2. 文件夹路径与变量定义
grid_folder      = r'D:\seoul\grids\lst_map'
target_vars      = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = [
    'BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)'
]

# 3. 主循环：遍历每个 *_clean.shp 并对每个 target 执行 LR 测试
for filename in os.listdir(grid_folder):
    if not filename.endswith('_clean.shp'):
        continue
    path = os.path.join(grid_folder, filename)
    print(f"\nProcessing {filename}")
    gdf = gpd.read_file(path)

    w   = libpysal.weights.Queen.from_dataframe(gdf)
    w.transform = 'r'
    n   = len(gdf)

    # 用来存放结果
    results = []

    for target in target_vars:
        # 构造 y, X
        y  = gdf[target].values.reshape(-1,1)
        X0 = gdf[explanatory_vars].values
        X  = np.hstack([np.ones((n,1)), X0])

        # 拟合 SDM（ML_Lag + slx_lags=1）
        model = ML_Lag(y, X, w=w, slx_lags=1)

        # SDM 的预测值
        y_pred = model.predy.flatten()
        y_true = y.flatten()

        # 计算 R²（使用定义 1 - SSE/SST）
        sse = np.sum((y_true - y_pred)**2)
        sst = np.sum((y_true - y_true.mean())**2)
        r2  = 1 - sse/sst

        # 计算 MAE 和 RMSE
        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        results.append({
            'Year':      target,
            'R_squared': round(r2,   4),
            'MAE':       round(mae,  4),
            'RMSE':      round(rmse, 4),
        })

    # 汇总成表
    metrics_df = pd.DataFrame(results).set_index('Year')
    print(metrics_df)


Processing city2020_lst_ratio_grid_1080m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp


C:\Users\owner\AppData\Local\Temp\ipykernel_40528\2466000845.py:30: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w   = libpysal.weights.Queen.from_dataframe(gdf)


          R_squared     MAE    RMSE
Year                               
nor_2020     0.9492  0.5046  0.6773
ext_2020     0.9579  0.6213  0.8357
hr_2020      0.9098  0.3090  0.4209

Processing city2020_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp


C:\Users\owner\AppData\Local\Temp\ipykernel_40528\2466000845.py:30: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w   = libpysal.weights.Queen.from_dataframe(gdf)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_lag.py:710: RuntimeWarning: divide by zero encountered in log
  jacob = np.log(np.linalg.det(a))
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_lag.py:710: RuntimeWarning: divide by zero encountered in log
  jacob = np.log(np.linalg.det(a))
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_lag.py:710: RuntimeWarning: divide by zero encountered in log
  jacob = np.log(np

In [ ]:

# 安装依赖（如未安装）
# pip install pysal geopandas pandas spreg scipy scikit-learn

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from spreg import ML_Lag
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 支持 libpysal 或 pysal.lib
try:
    import libpysal
    Queen = libpysal.weights.Queen
except ImportError:
    from pysal.lib.weights import Queen

def compute_sdm_metrics_for_distances(grid_folder, distances,
                                      explanatory_vars, target_vars):
    """
    Compute SDM metrics (R², MAE, RMSE) for shapefiles at given grid distances.

    Parameters:
      grid_folder: folder containing shapefiles.
      distances: list of integer distances (e.g. [360,450,720,1080]).
      explanatory_vars: list of explanatory variable column names.
      target_vars: list of target variable column names.

    Returns:
      DataFrame indexed by Distance and Year with columns [R_squared, MAE, RMSE].
    """
    results = []
    for dist in distances:
        pattern = f"_{dist}m_"
        # 找到对应距离的文件
        matches = [f for f in os.listdir(grid_folder)
                   if pattern in f and f.endswith('_clean.shp')]
        if not matches:
            continue
        filepath = os.path.join(grid_folder, matches[0])
        gdf = gpd.read_file(filepath)

        # 构造空间权重
        w = Queen.from_dataframe(gdf)
        w.transform = 'r'
        n = len(gdf)

        for target in target_vars:
            y_true = gdf[target].values
            X0 = gdf[explanatory_vars].values
            X = np.hstack([np.ones((n,1)), X0])

            # 估计 SDM
            model = ML_Lag(y_true.reshape(-1,1), X, w=w, slx_lags=1)
            y_pred = model.predy.flatten()

            # 计算指标
            sse = np.sum((y_true - y_pred)**2)
            sst = np.sum((y_true - y_true.mean())**2)
            r2  = 1 - sse/sst
            mae = mean_absolute_error(y_true, y_pred)
            rmse= np.sqrt(mean_squared_error(y_true, y_pred))

            results.append({
                'Distance':  dist,
                'Year':      target,
                'R_squared': round(r2, 4),
                'MAE':       round(mae, 4),
                'RMSE':      round(rmse, 4)
            })

    df = pd.DataFrame(results)
    return df.set_index(['Distance','Year'])


# —— 调用示例 ——
grid_folder = r'D:\seoul\grids\lst_map'
distances   = [360, 450, 720, 1080]
explanatory_vars = [
    'BCR(%)','BHV','NDVI','SVF','EV(m)',
    'Dist_BP','Dist_MT','Dist_WB','WR(%)'
]
target_vars = ['nor_2020','ext_2020','hr_2020']

metrics_df = compute_sdm_metrics_for_distances(
    grid_folder, distances, explanatory_vars, target_vars
)
print(metrics_df)


In [4]:
import requests
import pandas as pd

# 1. 构造请求
lat, lon = 40.7128, -74.0060  # 纽约
date = "20190830"             # 2019-08-30
url = (
    "https://power.larc.nasa.gov/api/temporal/hourly/point"
    f"?parameters=T2M,ALLSKY_SFC_SW_DWN,CLD"
    f"&community=RE&latitude={lat}&longitude={lon}"
    f"&start={date}&end={date}&format=JSON"
)

# 2. 获取并转换为 DataFrame
resp = requests.get(url)
js = resp.json()
df = pd.DataFrame(js['properties']['parameter'])
df.index = pd.to_datetime(df.index, format="%Y%m%d%H")  # 转为小时索引


KeyError: 'properties'

# 03 Random forest